In [1]:
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
from pathlib import Path
from scipy import sparse
import anndata
import seaborn as sns
import openpyxl

c:\Users\tesni\miniconda3\envs\ovarian-st\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_path = Path.cwd()

core_gene_list_path = current_path / "Tumor_core_bins.csv"
file_path_segmented = current_path / "A1" / "outs" / "segmented_outputs" / "filtered_feature_cell_matrix.h5"

In [3]:
with h5py.File(file_path_segmented, "r") as f:
    g = f["matrix"]
    data    = g["data"][:]
    indices = g["indices"][:]
    indptr  = g["indptr"][:]
    shape   = g["shape"][:]
    barcodes = g["barcodes"][:]
    genes    = g["features"]["name"][:]   # symbols

matrix = sparse.csc_matrix((data, indices, indptr), shape=shape)

adata_seg = anndata.AnnData(X=matrix.T)
adata_seg.obs_names = pd.Index(barcodes.astype(str))
adata_seg.var_names = pd.Index(genes.astype(str))

In [4]:
macrophage_markers_path = current_path / "macrophage_markers_list_reduced.xlsx"
macrophage_markers_df = pd.read_excel(macrophage_markers_path)
macrophage_markers_df

,Gene Symbol,Category / Function,Polarization / Subset,Association Confidence,Notes / Context
0,ADGRE1,Surface receptor,Tissue-resident,Strong,Classic macrophage lineage marker
1,APOE,Lipid-handling,M2/TAM,Strong,"Foam cells, lipid metabolism"
2,C1QA,Complement,General,Strong,Tissue-resident macrophages
3,C1QB,Complement,General,Strong,Tissue-resident macrophages
4,C1QC,Complement,General,Strong,Tissue-resident macrophages
...,...,...,...,...,...
86,TNFRSF1B,Receptor,General,Strong,TNF receptor
87,TNFSF13,Cytokine,Moderate,Moderate,Secreted by some macrophages
88,TREM1,Receptor,M1,Strong,Amplifies inflammatory responses
89,TREM2,Receptor,Tissue-resident,Strong,"Anti-inflammatory, tissue macrophages"


In [5]:
# 0) Make feature names unique
adata_seg.var_names_make_unique()   # critical line

# 1) Map marker symbols to the current var_names
marker_genes = macrophage_markers_df["Gene Symbol"].unique().tolist()
marker_genes_in_adata = adata_seg.var_names.intersection(marker_genes)
print(len(marker_genes_in_adata), "macrophage markers found")

# 2) Subset and run DE
adata_mac = adata_seg[:, marker_genes_in_adata].copy()

sc.pp.normalize_total(adata_mac, target_sum=1e4)
sc.pp.log1p(adata_mac)

sc.tl.rank_genes_groups(
    adata_mac,
    groupby="core_str",
    groups=["core"],
    reference="noncore",
    method="wilcoxon",
)

deg_mac = sc.get.rank_genes_groups_df(adata_mac, group="core")
deg_mac.head()

91 macrophage markers found


c:\Users\tesni\miniconda3\envs\ovarian-st\Lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


KeyError: 'core_str'

In [ ]:
# check that stats exist
deg_mac = sc.get.rank_genes_groups_df(adata_mac, group="core")
deg_mac

,names,scores,logfoldchanges,pvals,pvals_adj
0,CD9,15.600967,2.442628,7.169967e-55,3.262335e-53
1,C3,9.000372,0.996120,2.249541e-19,2.274536e-18
2,GAS6,8.469561,2.543823,2.463195e-17,1.867923e-16
3,IFITM3,7.416755,1.363259,1.200250e-13,7.801623e-13
4,CD55,5.638954,1.260713,1.710860e-08,1.037921e-07
5,ISG15,3.697216,2.027744,2.179768e-04,9.016315e-04
6,STAT1,3.469406,1.654189,5.216105e-04,2.063763e-03
7,INHBA,3.043947,2.714749,2.334965e-03,8.304757e-03
8,IRF1,0.872629,1.031303,3.828652e-01,7.801054e-01
9,PROS1,0.820710,1.069477,4.118115e-01,7.815017e-01


In [ ]:
sig = deg_mac.query("pvals_adj < 0.05").sort_values("logfoldchanges", ascending=False)
sig.head(20)

,names,scores,logfoldchanges,pvals,pvals_adj
7,INHBA,3.043947,2.714749,2.334965e-03,8.304757e-03
2,GAS6,8.469561,2.543823,2.463195e-17,1.867923e-16
0,CD9,15.600967,2.442628,7.169967e-55,3.262335e-53
5,ISG15,3.697216,2.027744,2.179768e-04,9.016315e-04
6,STAT1,3.469406,1.654189,5.216105e-04,2.063763e-03
3,IFITM3,7.416755,1.363259,1.200250e-13,7.801623e-13
4,CD55,5.638954,1.260713,1.710860e-08,1.037921e-07
1,C3,9.000372,0.996120,2.249541e-19,2.274536e-18
68,CTSB,-2.581761,-0.252000,9.829769e-03,2.885513e-02
79,SPP1,-5.462204,-0.498845,4.702609e-08,2.517279e-07
